# Prepare_contract_year

This notebook prepares the contract-year base table used throughout the thesis.

Goal:
- start from raw contract metadata
- expand contracts to one row per contract-year
- repair missing department values using a cost-center mapping
- run compact sanity checks
- save the final contract-year table to `Data/raw/contract_year_final.csv`

## Imports and project path setup

In [115]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

project_root: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-
src_path: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/src


In [116]:
import pandas as pd
import numpy as np

## Load raw contract metadata

In [117]:
raw_contract_path = project_root / "Data" / "raw" / "dim_ci_stso_contracts_metadata.csv"

df_contracts = pd.read_csv(
    raw_contract_path,
    engine="python",
    on_bad_lines="skip",
    #low_memory=False,
)

print("Shape:", df_contracts.shape)
print("Unique contract_id:", df_contracts["contract_id"].nunique())
print("Unique contract_number:", df_contracts["contract_number"].nunique())

display(df_contracts.head())

Shape: (2252, 97)
Unique contract_id: 2252
Unique contract_number: 2252


,contract_id,contract_number,contract_name,contract_status,terminated,term_type,contract_owner,contract_owner_cost_centre,start_date,expiration_date,...,legal_agreement_file_name,is_the_contract_gxp_regulated,attachments_names,supplier_display_name,Preferred_supplier_tag,unit,department,team,days_until_expiry,expiry_range
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,RSRV,7756.0,2018-05-21T00:00:00.000Z,2027-07-30T00:00:00.000Z,...,NaN,Yes,Bioreliance_MSA_2018.pdf,BIORELIANCE LIMITED INVITROGEN BIOSERVICES,0.0,SSIMS,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",516.0,Over 360 days
1,8157,8158,Kalundborg Bioenergi_Master_2017_SA,published,False,fixed,BLHN,1751.0,2018-04-01T00:00:00.000Z,2033-07-01T00:00:00.000Z,...,NaN,NaN,Kal_Bioenergi_Purchase_Agreement_2017.03.16.pdf,KALUNDBORG BIOENERGI APS,0.0,SSIMS,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",2679.0,Over 360 days
2,1276,1276,Lonza Ltd & Lonza Sales Ltd_Master_2017_FA,published,False,auto_renew,MKYR,1751.0,2017-02-22T00:00:00.000Z,2037-02-22T00:00:00.000Z,...,Lonza_Ltd___Lonza_Sales_Ltd_Master_20170222_FA...,NaN,Lonza_Ltd___Lonza_Sales_Ltd_Internal_doc_20170...,LONZA SALES LTD,0.0,NaN,NaN,NaN,4011.0,Over 360 days
3,201,201,Evonik_Master_20160520_SA,published,False,fixed,PALD,1751.0,2016-05-01T00:00:00.000Z,2026-05-01T00:00:00.000Z,...,Evonik_Master_20160520_SA.pdf,NaN,Evonik_Internal_Doc_20160520_CL.pdf;Evonik_Cal...,"EVONIK REXIM (NANNING) PHARMACEUTIC CO., LTD",0.0,NaN,NaN,NaN,61.0,90 days
4,19779,19779,DONG ENERGY THERMAL POWER A/S_Master_2017_SA_S...,published,False,fixed,ENOM,1751.0,2017-06-22T00:00:00.000Z,2039-12-31T00:00:00.000Z,...,DONG_Energy_Thermal_Power_A_S_SA_20170623.PDF,NaN,DONG_Energy_Thermal_Power_A_S_SA_APP_20170623_...,DONG ENERGY THERMAL POWER A/S,0.0,SSIMS,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",5053.0,Over 360 days


## Quick checks on identifiers

In [118]:
print("Duplicate contract_id rows:", df_contracts["contract_id"].duplicated().sum())
print("Duplicate contract_number rows:", df_contracts["contract_number"].duplicated().sum())

Duplicate contract_id rows: 0
Duplicate contract_number rows: 0


In [119]:
df_contracts["contract_status"].value_counts(dropna=False)

contract_status
published                      2251
F060 Invoice Date + 60 days       1
Name: count, dtype: int64

In [120]:
df_contracts["terminated"].value_counts(dropna=False)

terminated
False    2250
NaN         2
Name: count, dtype: int64

In [121]:
tmp = df_contract_base.copy()

analysis_year_min = 2015
analysis_year_max = 2025

tmp["end_year"] = tmp["expiry_year"].clip(upper=analysis_year_max)
tmp["start_year_capped"] = tmp["start_year"].clip(lower=analysis_year_min)

tmp["passes_window"] = (
    tmp["start_year_capped"].notna()
    & tmp["end_year"].notna()
    & (tmp["start_year_capped"] <= tmp["end_year"])
)

dropped = tmp.loc[~tmp["passes_window"]].copy()

print("Dropped contracts:", len(dropped))

display(
    dropped[
        [
            "contract_id",
            "contract_number",
            "start_date",
            "expiration_date",
            "start_year",
            "expiry_year",
        ]
    ]
    .sort_values(["start_year", "expiry_year"])
)

Dropped contracts: 0


,contract_id,contract_number,start_date,expiration_date,start_year,expiry_year


## Parse date columns

In [122]:
date_cols = ["start_date", "expiration_date", "execution_at", "published_at"]

for col in date_cols:
    if col in df_contracts.columns:
        df_contracts[col] = pd.to_datetime(df_contracts[col], errors="coerce")

print("Start date min:", df_contracts["start_date"].min())
print("Start date max:", df_contracts["start_date"].max())
print("Expiry date min:", df_contracts["expiration_date"].min())
print("Expiry date max:", df_contracts["expiration_date"].max())

display(df_contracts[["start_date", "expiration_date"]].isna().mean().to_frame("missing_pct"))

Start date min: 2001-01-10 00:00:00+00:00
Start date max: 2026-03-01 00:00:00+00:00
Expiry date min: 2026-03-01 00:00:00+00:00
Expiry date max: 2099-12-31 00:00:00+00:00


,missing_pct
start_date,0.000444
expiration_date,0.000444


In [123]:
missing_contracts = ['1877', '25777', '346312', '536319', '598142', '612555']

metadata_contract_numbers = set(
    df_contracts["contract_number"]
    .astype(str)
    .str.strip()
)

metadata_contract_ids = set(
    df_contracts["contract_id"]
    .astype(str)
    .str.strip()
)

for c in missing_contracts:

    print(
        c,
        "| in contract_number:", c in metadata_contract_numbers,
        "| in contract_id:", c in metadata_contract_ids
    )

1877 | in contract_number: False | in contract_id: False
25777 | in contract_number: False | in contract_id: False
346312 | in contract_number: False | in contract_id: False
536319 | in contract_number: False | in contract_id: False
598142 | in contract_number: False | in contract_id: False
612555 | in contract_number: False | in contract_id: False


In [124]:
missing_contracts = ['1877', '25777', '346312', '536319', '598142', '612555']

df_contracts["contract_id"] = df_contracts["contract_id"].astype(str).str.strip()
df_contracts["contract_number"] = df_contracts["contract_number"].astype(str).str.strip()

check_contract_number = df_contracts[df_contracts["contract_number"].isin(missing_contracts)].copy()

print("Found as contract_number:", check_contract_number["contract_number"].nunique(), "out of", len(missing_contracts))
print("Numbers found:", sorted(check_contract_number["contract_number"].unique()))

display(
    check_contract_number[
        ["contract_id", "contract_number", "contract_name", "start_date", "expiration_date", "contract_status"]
    ].sort_values("contract_number")
)

Found as contract_number: 0 out of 6
Numbers found: []


,contract_id,contract_number,contract_name,start_date,expiration_date,contract_status


## Build contract-level working table

In [125]:
keep_cols = [
    "contract_id",
    "contract_number",
    "contract_name",
    "contract_status",
    "contract_owner_cost_centre",
    "terminated",
    "term_type",
    "start_date",
    "expiration_date",
    "supplier_id",
    "supplier_number",
    "supplier_display_name",
    "payment_terms",
    "incoterms",
    "contract_currency_code",
    "contract_value",
    "contract_value_dkk",
    "contract_type",
    "nn_contract_type",
    "contract_commodity",
    "team",
    "unit",
    "department",
    "company_country",
]

keep_cols = [c for c in keep_cols if c in df_contracts.columns]

df_contract_base = df_contracts[keep_cols].copy()

print("Shape:", df_contract_base.shape)
display(df_contract_base.head())

Shape: (2252, 24)


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,contract_currency_code,contract_value,contract_value_dkk,contract_type,nn_contract_type,contract_commodity,team,unit,department,company_country
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,...,GBP,0.0000,0.0,Production,Master Service Agreement,Outsourced GMP Laboratory Analysis,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK
1,8157,8158,Kalundborg Bioenergi_Master_2017_SA,published,1751.0,False,fixed,2018-04-01 00:00:00+00:00,2033-07-01 00:00:00+00:00,17030.0,...,DKK,0.0000,0.0,Production,NaN,Waste Management,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK
2,1276,1276,Lonza Ltd & Lonza Sales Ltd_Master_2017_FA,published,1751.0,False,auto_renew,2017-02-22 00:00:00+00:00,2037-02-22 00:00:00+00:00,13689.0,...,CHF,0.0000,0.0,Production,NaN,Outsourced Product Services,NaN,NaN,NaN,DENMARK
3,201,201,Evonik_Master_20160520_SA,published,1751.0,False,fixed,2016-05-01 00:00:00+00:00,2026-05-01 00:00:00+00:00,16213.0,...,DKK,0.0000,0.0,Production,Other,Excipients,NaN,NaN,NaN,DENMARK
4,19779,19779,DONG ENERGY THERMAL POWER A/S_Master_2017_SA_S...,published,1751.0,False,fixed,2017-06-22 00:00:00+00:00,2039-12-31 00:00:00+00:00,16440.0,...,DKK,0.0000,0.0,Production,Master Service Agreement,Fuel,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK


## Derive contract-year boundaries

In [126]:
df_contract_base["start_year"] = df_contract_base["start_date"].dt.year
df_contract_base["expiry_year"] = df_contract_base["expiration_date"].dt.year

# Treat far-future expiries as open-ended agreements
df_contract_base["open_ended_contract"] = (df_contract_base["expiry_year"] >= 2090).astype(int)

display(
    df_contract_base[
        ["contract_id", "contract_number", "start_date", "expiration_date", "start_year", "expiry_year", "open_ended_contract"]
    ].head()
)

,contract_id,contract_number,start_date,expiration_date,start_year,expiry_year,open_ended_contract
0,9675,9675,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,2018.0,2027.0,0
1,8157,8158,2018-04-01 00:00:00+00:00,2033-07-01 00:00:00+00:00,2018.0,2033.0,0
2,1276,1276,2017-02-22 00:00:00+00:00,2037-02-22 00:00:00+00:00,2017.0,2037.0,0
3,201,201,2016-05-01 00:00:00+00:00,2026-05-01 00:00:00+00:00,2016.0,2026.0,0
4,19779,19779,2017-06-22 00:00:00+00:00,2039-12-31 00:00:00+00:00,2017.0,2039.0,0


## Define analysis window

In [127]:
analysis_year_min = 2015
analysis_year_max = 2025

df_contract_base["end_year"] = df_contract_base["expiry_year"].clip(upper=analysis_year_max)
df_contract_base["start_year_capped"] = df_contract_base["start_year"].clip(lower=analysis_year_min)

df_contract_base = df_contract_base[
    df_contract_base["start_year_capped"].notna()
    & df_contract_base["end_year"].notna()
    & (df_contract_base["start_year_capped"] <= df_contract_base["end_year"])
].copy()

print("Shape after window filter:", df_contract_base.shape)

Shape after window filter: (2209, 29)


## Expand to one row per contract-year

In [128]:
df_contract_base["year_list"] = df_contract_base.apply(
    lambda row: list(range(int(row["start_year_capped"]), int(row["end_year"]) + 1)),
    axis=1,
)

df_contract_year = (
    df_contract_base
    .explode("year_list")
    .rename(columns={"year_list": "observation_year"})
    .reset_index(drop=True)
)

df_contract_year["observation_year"] = pd.to_numeric(
    df_contract_year["observation_year"],
    errors="coerce"
).astype("Int64")

print("Contract-year shape:", df_contract_year.shape)
display(df_contract_year.head())

Contract-year shape: (9201, 30)


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,team,unit,department,company_country,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,...,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK,2018.0,2027.0,0,2025.0,2018.0,2018
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,...,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK,2018.0,2027.0,0,2025.0,2018.0,2019
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,...,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK,2018.0,2027.0,0,2025.0,2018.0,2020
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,...,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK,2018.0,2027.0,0,2025.0,2018.0,2021
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,...,"Quality, Production Services & Supplies",SSIMS,"Quality, Production Services & Supplies",DENMARK,2018.0,2027.0,0,2025.0,2018.0,2022


## Create time-dependent contract features

In [129]:
df_contract_year["years_to_expiry"] = df_contract_year["expiry_year"] - df_contract_year["observation_year"]
df_contract_year["contract_age_years"] = df_contract_year["observation_year"] - df_contract_year["start_year"]

df_contract_year["expiry_pressure_bucket"] = pd.cut(
    df_contract_year["years_to_expiry"],
    bins=[-np.inf, 0, 1, 3, 5, np.inf],
    labels=["expired_or_expiring", "within_1y", "1_to_3y", "3_to_5y", "5y_plus"],
)

## Standardize identifiers and numeric types

In [130]:
df_contract_year["contract_id"] = df_contract_year["contract_id"].astype("string")
df_contract_year["contract_number"] = df_contract_year["contract_number"].astype("string")
df_contract_year["supplier_number"] = df_contract_year["supplier_number"].astype("string")

if "supplier_id" in df_contract_year.columns:
    df_contract_year["supplier_id"] = pd.to_numeric(
        df_contract_year["supplier_id"],
        errors="coerce"
    ).astype("Int64")

year_cols = [
    "start_year",
    "expiry_year",
    "end_year",
    "start_year_capped",
    "observation_year",
    "years_to_expiry",
    "contract_age_years",
]
for col in year_cols:
    if col in df_contract_year.columns:
        df_contract_year[col] = pd.to_numeric(df_contract_year[col], errors="coerce").astype("Int64")

## Check and repair missing department assignments

In [131]:
print("Unique contracts without department:", df_contract_year.loc[df_contract_year["department"].isna(), "contract_id"].nunique())
print("Unique missing cost centers:", df_contract_year.loc[df_contract_year["department"].isna(), "contract_owner_cost_centre"].nunique())

display(
    df_contract_year.loc[
        df_contract_year["department"].isna(),
        ["contract_id", "contract_name", "contract_owner_cost_centre", "department", "team", "unit"]
    ].drop_duplicates().head(20)
)

Unique contracts without department: 607
Unique missing cost centers: 9


,contract_id,contract_name,contract_owner_cost_centre,department,team,unit
16,1276,Lonza Ltd & Lonza Sales Ltd_Master_2017_FA,1751.0,NaN,NaN,NaN
25,201,Evonik_Master_20160520_SA,1751.0,NaN,NaN,NaN
44,124869,Boehringer Ingelheim Pharma GmbH & Co. KG_Call...,1751.0,NaN,NaN,NaN
68,8197,NOF Corporation_Master_20150309_SA,1751.0,NaN,NaN,NaN
90,209179,Kuehne_Nagel_Call-off_2022_NNAS_Primary_4PL_Am...,1750.0,NaN,NaN,NaN
94,190480,Kuehne_Nagel_Call-off_2021_NNAS_Primary_4PL_Am...,1750.0,NaN,NaN,NaN
117,97473,Kuehne Nagel_Call-off_2020_NNAS_Primary_Amendm...,1750.0,NaN,NaN,NaN
123,123436,Harman Finochem Ltd_Stand alone_2021_SA,280.0,NaN,NaN,NaN
140,94706,F.I.S._Enantia_Stand-alone_2020_CDA_3 way_SNAC,1751.0,NaN,NaN,NaN
242,7651,Boehringer Ingelheim Pharma GmbH_Master_201111...,1751.0,NaN,NaN,NaN


In [132]:
cost_center_department_map = {
    1751: "Drug Substance Outsourcing",
    1750: "Devices & Needles",
    280: "Global Strategic Outsourcing & Devices",
    448: "HI Warehouse Expansion Program",
    8377: "Drug Product Outsourcing",
    1753: "Bioprocessing & Raw Materials",
    1756: "Bioprocessing & Excipients",
    1755: "Bioprocessing and Excipients",
    8375: "Quality Control",
}

In [133]:
df_contract_year["department_from_cost_center"] = df_contract_year["contract_owner_cost_centre"].map(cost_center_department_map)

df_contract_year["department_filled"] = df_contract_year["department"]
df_contract_year.loc[
    df_contract_year["department_filled"].isna(),
    "department_filled"
] = df_contract_year.loc[
    df_contract_year["department_filled"].isna(),
    "department_from_cost_center"
]

In [134]:
print("Unique contracts still missing department after fill:", df_contract_year.loc[df_contract_year["department_filled"].isna(), "contract_id"].nunique())

display(
    df_contract_year.loc[
        df_contract_year["department"].isna(),
        ["contract_id", "contract_name", "contract_owner_cost_centre", "department", "department_from_cost_center", "department_filled"]
    ].drop_duplicates().head(20)
)

Unique contracts still missing department after fill: 22


,contract_id,contract_name,contract_owner_cost_centre,department,department_from_cost_center,department_filled
16,1276,Lonza Ltd & Lonza Sales Ltd_Master_2017_FA,1751.0,NaN,Drug Substance Outsourcing,Drug Substance Outsourcing
25,201,Evonik_Master_20160520_SA,1751.0,NaN,Drug Substance Outsourcing,Drug Substance Outsourcing
44,124869,Boehringer Ingelheim Pharma GmbH & Co. KG_Call...,1751.0,NaN,Drug Substance Outsourcing,Drug Substance Outsourcing
68,8197,NOF Corporation_Master_20150309_SA,1751.0,NaN,Drug Substance Outsourcing,Drug Substance Outsourcing
90,209179,Kuehne_Nagel_Call-off_2022_NNAS_Primary_4PL_Am...,1750.0,NaN,Devices & Needles,Devices & Needles
94,190480,Kuehne_Nagel_Call-off_2021_NNAS_Primary_4PL_Am...,1750.0,NaN,Devices & Needles,Devices & Needles
117,97473,Kuehne Nagel_Call-off_2020_NNAS_Primary_Amendm...,1750.0,NaN,Devices & Needles,Devices & Needles
123,123436,Harman Finochem Ltd_Stand alone_2021_SA,280.0,NaN,Global Strategic Outsourcing & Devices,Global Strategic Outsourcing & Devices
140,94706,F.I.S._Enantia_Stand-alone_2020_CDA_3 way_SNAC,1751.0,NaN,Drug Substance Outsourcing,Drug Substance Outsourcing
242,7651,Boehringer Ingelheim Pharma GmbH_Master_201111...,1751.0,NaN,Drug Substance Outsourcing,Drug Substance Outsourcing


## Finalize department column

In [135]:
df_contract_year = (
    df_contract_year
    .drop(columns=["department"], errors="ignore")
    .rename(columns={"department_filled": "department"})
)

print("Shape:", df_contract_year.shape)
display(df_contract_year.head())

Shape: (9201, 34)


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,expiry_year,open_ended_contract,end_year,start_year_capped,observation_year,years_to_expiry,contract_age_years,expiry_pressure_bucket,department_from_cost_center,department
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2018,9,0,5y_plus,NaN,"Quality, Production Services & Supplies"
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2019,8,1,5y_plus,NaN,"Quality, Production Services & Supplies"
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2020,7,2,5y_plus,NaN,"Quality, Production Services & Supplies"
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2021,6,3,5y_plus,NaN,"Quality, Production Services & Supplies"
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,2027,0,2025,2018,2022,5,4,3_to_5y,NaN,"Quality, Production Services & Supplies"


## Final sanity checks

In [136]:
print("Final shape:", df_contract_year.shape)
print("Unique contracts:", df_contract_year["contract_id"].nunique())
print("Unique suppliers:", df_contract_year["supplier_id"].nunique() if "supplier_id" in df_contract_year.columns else "supplier_id not present")
print("Unique departments:", df_contract_year["department"].nunique(dropna=True))
print("Duplicate contract-year rows:", df_contract_year.duplicated(["contract_id", "observation_year"]).sum())
print("Missing contract_id:", df_contract_year["contract_id"].isna().sum())
print("Missing observation_year:", df_contract_year["observation_year"].isna().sum())
print("Missing department:", df_contract_year["department"].isna().sum())

Final shape: (9201, 34)
Unique contracts: 2209
Unique suppliers: 583
Unique departments: 14
Duplicate contract-year rows: 0
Missing contract_id: 0
Missing observation_year: 0
Missing department: 92


In [137]:
df_contracts_per_department = (
    df_contract_year
    .groupby("department", dropna=False)["contract_id"]
    .nunique()
    .reset_index(name="unique_contract_count")
    .sort_values("unique_contract_count", ascending=False)
)

df_rows_per_department = (
    df_contract_year
    .groupby("department", dropna=False)
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

display(df_contracts_per_department)
display(df_rows_per_department)

,department,unique_contract_count
3,Devices & Needles,476
4,Drug Product Outsourcing,300
5,Drug Substance Outsourcing,298
9,Packaging Material,289
11,Raw Materials & Energy,281
10,"Quality, Production Services & Supplies",215
1,Bioprocessing & Raw Materials,113
8,Logistics,94
2,Bioprocessing and Excipients,59
0,"Alliance, Acquisitions & PPM CoE",35


,department,row_count
3,Devices & Needles,2197
11,Raw Materials & Energy,1555
9,Packaging Material,1272
10,"Quality, Production Services & Supplies",1044
5,Drug Substance Outsourcing,943
1,Bioprocessing & Raw Materials,679
4,Drug Product Outsourcing,589
8,Logistics,282
0,"Alliance, Acquisitions & PPM CoE",270
2,Bioprocessing and Excipients,218


In [138]:
print("Observation year range:", df_contract_year["observation_year"].min(), "to", df_contract_year["observation_year"].max())

display(
    df_contract_year["observation_year"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("row_count")
)

Observation year range: 2015 to 2025


,row_count
observation_year,
2015,192
2016,226
2017,262
2018,331
2019,438
2020,578
2021,807
2022,1130
2023,1349


## Save final contract-year table

In [139]:
output_path = project_root / "Data" / "raw" / "contract_year_final.csv"

df_contract_year.to_csv(output_path, index=False)

print("Saved file to:", output_path)
print("Final shape:", df_contract_year.shape)
print("Unique contracts:", df_contract_year["contract_id"].nunique())
print("Unique departments:", df_contract_year["department"].nunique(dropna=True))

Saved file to: /Users/annita/Desktop/Thesis/Signal_Fusion_with_Meta-learners-/Data/raw/contract_year_final.csv
Final shape: (9201, 34)
Unique contracts: 2209
Unique departments: 14
